# 🎙️ P2 — Speech AI Engineer (v2 — intégration P1)
## Speech-to-Retrieval System (SRS) — INPT Project

**Objectif** : Transformer des fichiers audio en embeddings (dim=768)  
**Nouveauté v2** : Intégration directe des outputs de P1 (`audio_manifest.csv`, `pairs.csv`)

---
### Plan
1. Installation & Setup
2. Chargement des données de P1
3. Modèle Wav2Vec2
4. Fonction `speech_to_embedding()`
5. Batch encoding de tous les audios P1
6. Sauvegarde `audio_embeddings.npy` + `audio_embeddings_index.csv`
7. Caching & optimisation
8. SpeechEncoder class (pour P3)
9. Export ONNX (pour P4)
10. Tests système


In [1]:
!pip install -q transformers accelerate torchaudio librosa soundfile onnx onnxruntime joblib tqdm mlflow
print('✅ Installation OK')

✅ Installation OK


## ✅ CELLULE 2 — Imports & Config

In [2]:
import os, json, time, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import librosa
import soundfile as sf
import joblib
from pathlib import Path
from tqdm import tqdm
from transformers import Wav2Vec2Processor, Wav2Vec2Model

warnings.filterwarnings('ignore')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'🖥️  Device : {DEVICE}')
if DEVICE == 'cuda':
    print(f'🎮  GPU    : {torch.cuda.get_device_name(0)}')

CONFIG = {
    'model_name'       : 'facebook/wav2vec2-base-960h',
    # Pour multilingue (Darija/FR) : 'facebook/wav2vec2-xls-r-300m'
    'sample_rate'      : 16000,
    'embedding_dim'    : 768,
    'pooling_strategy' : 'mean',
    'batch_size'       : 8,
    'min_duration'     : 1.0,
    'max_duration'     : 20.0,

    # ── Paths issus de P1 ──────────────────────────────────
    'p1_audio_dir'     : '../data/audio_clean/',        # audios propres de P1
    'p1_manifest'      : '../data/output/audio_manifest.csv', # index P1
    'p1_pairs'         : '../data/output/pairs.csv',   # paires P1
    'p1_pairs_train'   : '../data/output/pairs_train.csv',
    'p1_pairs_val'     : '../data/output/pairs_val.csv',
    'p1_pairs_test'    : '../data/output/pairs_test.csv',

    # ── Outputs P2 ─────────────────────────────────────────
    'embeddings_dir'   : '../data/embeddings/',
    'embeddings_matrix': '../data/embeddings/audio_embeddings.npy',
    'embeddings_index' : '../data/embeddings/audio_embeddings_index.csv',
    'cache_file'       : '../data/embeddings/cache.pkl',
    'onnx_path'        : './models/speech_encoder.onnx',
}

for p in [CONFIG['embeddings_dir'], './models']:
    os.makedirs(p, exist_ok=True)

print('\n⚙️  Configuration chargée')

c:\Users\mrtds\Documents\ProjetPython\AudioProcessing\inpt_audioprocessing\mamadou\myenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🖥️  Device : cpu

⚙️  Configuration chargée


## ✅ CELLULE 3 — Charger les données de P1

In [3]:
# ════════════════════════════════════════════════════════════
# CHARGEMENT DES OUTPUTS DE P1
# ════════════════════════════════════════════════════════════

def load_p1_data(config: dict) -> tuple:
    """
    Charge les fichiers produits par P1.

    Returns:
        (manifest_df, pairs_df, train_df, val_df, test_df)
    """
    loaded = {}

    for name, path in [
        ('manifest', config['p1_manifest']),
        ('pairs',    config['p1_pairs']),
        ('train',    config['p1_pairs_train']),
        ('val',      config['p1_pairs_val']),
        ('test',     config['p1_pairs_test']),
    ]:
        if os.path.exists(path):
            loaded[name] = pd.read_csv(path)
            print(f'   ✅ {name:10s} : {len(loaded[name]):5d} lignes  ← {path}')
        else:
            print(f'   ⚠️  {name:10s} : fichier introuvable → {path}')
            print(f'       (lancer P1 d\'abord ou vérifier le chemin)')
            loaded[name] = pd.DataFrame()

    return loaded


print('📂 Chargement données P1...')
p1_data = load_p1_data(CONFIG)

manifest_df = p1_data['manifest']
pairs_df    = p1_data['pairs']
train_df    = p1_data['train']
val_df      = p1_data['val']
test_df     = p1_data['test']

if not manifest_df.empty:
    print(f'\n📊 Dataset P1 :')
    print(f'   Fichiers audio : {len(manifest_df)}')
    print(f'   Durée totale   : {manifest_df["duration"].sum()/3600:.2f}h')
    print(f'   Sources        : {manifest_df["source"].value_counts().to_dict()}')
    print(manifest_df.head(3))
else:
    print('\n⚠️  Données P1 non trouvées — mode test avec audio synthétique')

📂 Chargement données P1...
   ✅ manifest   :  4985 lignes  ← ../data/output/audio_manifest.csv
   ✅ pairs      :  4985 lignes  ← ../data/output/pairs.csv
   ✅ train      :  3988 lignes  ← ../data/output/pairs_train.csv
   ✅ val        :   498 lignes  ← ../data/output/pairs_val.csv
   ✅ test       :   499 lignes  ← ../data/output/pairs_test.csv

📊 Dataset P1 :
   Fichiers audio : 4985
   Durée totale   : 17.61h
   Sources        : {'librispeech_train.100': 3000, 'librispeech_train.360': 1985}
         audio_id            filename                               filepath  \
0  libri100_01383  libri100_01383.wav  ../data/audio_clean/libri100_01383.wav   
1  libri100_01881  libri100_01881.wav  ../data/audio_clean/libri100_01881.wav   
2  libri360_01901  libri360_01901.wav  ../data/audio_clean/libri360_01901.wav   

                                                text                 source  \
0  WHEN HE TURNED FOR RUTH SHE HAD ALREADY SPRUNG...  librispeech_train.100   
1  AND THUS THAT CHUR

## ✅ CELLULE 4 — Chargement modèle Wav2Vec2

In [4]:
print(f"⏳ Chargement : {CONFIG['model_name']}")

processor = Wav2Vec2Processor.from_pretrained(CONFIG['model_name'])
model     = Wav2Vec2Model.from_pretrained(CONFIG['model_name']).to(DEVICE).eval()

total  = sum(p.numel() for p in model.parameters())
frozen = sum(p.numel() for p in model.parameters() if not p.requires_grad)
print(f'✅ Modèle chargé sur {DEVICE}')
print(f'   Paramètres : {total:,}  (gelés : {frozen:,})')

⏳ Chargement : facebook/wav2vec2-base-960h


Loading weights: 100%|██████████| 210/210 [00:00<00:00, 3055.29it/s]
Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base-960h
Key               | Status     | 
------------------+------------+-
lm_head.weight    | UNEXPECTED | 
lm_head.bias      | UNEXPECTED | 
masked_spec_embed | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✅ Modèle chargé sur cpu
   Paramètres : 94,371,712  (gelés : 0)


## ✅ CELLULE 5 — Preprocessing & speech_to_embedding()

In [5]:
from typing import Optional, Tuple
import tempfile

def load_audio(path: str) -> Optional[np.ndarray]:
    """Charge un fichier audio → numpy 16kHz mono normalisé."""
    try:
        wav, _ = librosa.load(path, sr=CONFIG['sample_rate'], mono=True)
        dur = len(wav) / CONFIG['sample_rate']
        if dur < CONFIG['min_duration']:  return None
        if dur > CONFIG['max_duration']:
            wav = wav[:int(CONFIG['max_duration'] * CONFIG['sample_rate'])]
        peak = np.abs(wav).max()
        return wav / peak if peak > 0 else wav
    except Exception as e:
        print(f'⚠️  Erreur chargement audio : {e}')
        return None


def speech_to_embedding(audio_input,
                         normalize: bool = True) -> Optional[np.ndarray]:
    """
    Convertit un fichier audio en vecteur d'embedding (768,).

    Args:
        audio_input : str (chemin .wav) ou np.ndarray (waveform)
        normalize   : normalisation L2 (recommandé pour cosine similarity)

    Returns:
        np.ndarray (768,) ou None si erreur
    """
    # 1. Chargement
    if isinstance(audio_input, str):
        wav = load_audio(audio_input)
    elif isinstance(audio_input, np.ndarray):
        wav = audio_input.astype(np.float32)
    else:
        print(f'❌ Type non supporté : {type(audio_input)}')
        return None

    if wav is None: return None

    # 2. Preprocessing — s'assurer que c'est float32
    wav = wav.astype(np.float32)
    inp = processor(
        wav,
        sampling_rate=CONFIG['sample_rate'],
        return_tensors='pt',
        padding=True
    ).input_values.to(DEVICE)

    # 3. Forward
    with torch.no_grad():
        hidden = model(inp).last_hidden_state  # [1, T, 768]

    # 4. Mean pooling
    emb = hidden.mean(dim=1).squeeze(0).cpu().numpy()  # (768,)

    # 5. Normalisation L2
    if normalize:
        norm = np.linalg.norm(emb)
        if norm > 0: emb = emb / norm

    return emb


print('✅ Fonctions définies : load_audio(), speech_to_embedding()')


✅ Fonctions définies : load_audio(), speech_to_embedding()


## ✅ CELLULE 6 — Batch encoding COMPLET des audios de P1

In [6]:
# ════════════════════════════════════════════════════════════
# BATCH ENCODING — Encoder TOUS les audios de P1
# Produit : audio_embeddings.npy + audio_embeddings_index.csv
# ════════════════════════════════════════════════════════════

def encode_all_p1_audios(manifest_df: pd.DataFrame,
                           config: dict):
    """
    Encode tous les fichiers audio du manifest P1.

    Returns:
        embeddings_matrix : np.ndarray [N, 768]
        index_df          : DataFrame avec audio_id, filepath, embedding_row
    """
    if manifest_df.empty:
        print('⚠️  manifest_df vide — génération de données synthétiques pour test')
        N = 50
        embeddings = np.random.randn(N, 768).astype(np.float32)
        embeddings /= np.linalg.norm(embeddings, axis=1, keepdims=True)
        idx = pd.DataFrame({
            'audio_id'     : [f'test_{i:04d}' for i in range(N)],
            'filename'     : [f'test_{i:04d}.wav' for i in range(N)],
            'embedding_row': list(range(N)),
            'status'       : 'synthetic',
        })
        return embeddings, idx

    audio_files = manifest_df['filepath'].tolist()
    audio_ids   = manifest_df['audio_id'].tolist()
    batch_size  = config['batch_size']

    # Longueur max en samples pour le padding (éviter OOM)
    MAX_LENGTH  = int(config['max_duration'] * config['sample_rate'])  # 20s × 16000

    all_embeddings, index_rows = [], []
    row_counter = 0

    print(f'🎙️  Encoding {len(audio_files)} fichiers audio (batch={batch_size})...')

    for i in tqdm(range(0, len(audio_files), batch_size), desc='Batch encoding'):
        batch_paths = audio_files[i:i+batch_size]
        batch_ids   = audio_ids[i:i+batch_size]

        waveforms, valid_ids, valid_paths = [], [], []

        for fpath, fid in zip(batch_paths, batch_ids):
            wav = load_audio(fpath)
            if wav is not None:
                # Garantir float32 et longueur <= MAX_LENGTH
                wav = wav.astype(np.float32)[:MAX_LENGTH]
                waveforms.append(wav)
                valid_ids.append(fid)
                valid_paths.append(fpath)

        if not waveforms:
            continue

        try:
            inp = processor(
                waveforms,
                sampling_rate=config['sample_rate'],
                return_tensors='pt',
                padding=True,
                truncation=True,                  # ← FIX : évite OOM sur audio très long
                max_length=MAX_LENGTH,            # ← FIX : plafond explicite
                return_attention_mask=True
            ).input_values.to(DEVICE)

            with torch.no_grad():
                hidden = model(inp).last_hidden_state  # [B, T, 768]
            embs = hidden.mean(dim=1).cpu().numpy()    # [B, 768]

            # Normalisation L2
            norms = np.linalg.norm(embs, axis=1, keepdims=True)
            embs  = embs / np.maximum(norms, 1e-8)

            for fid, fpath, emb in zip(valid_ids, valid_paths, embs):
                all_embeddings.append(emb)
                index_rows.append({
                    'audio_id'     : fid,
                    'filepath'     : fpath,
                    'embedding_row': row_counter,
                    'status'       : 'ok',
                })
                row_counter += 1

        except RuntimeError as e:
            if 'out of memory' in str(e).lower():
                torch.cuda.empty_cache()
                print(f'\n⚠️  OOM — réduire batch_size dans CONFIG (actuel: {batch_size})')
            else:
                print(f'\n❌ Erreur batch {i}: {e}')

    embeddings_matrix = np.array(all_embeddings, dtype=np.float32)  # [N, 768]
    index_df          = pd.DataFrame(index_rows)

    # Sauvegarder
    np.save(config['embeddings_matrix'], embeddings_matrix)
    index_df.to_csv(config['embeddings_index'], index=False)

    print(f'\n✅ Encoding terminé !')
    print(f'   Matrix : {embeddings_matrix.shape}  → {config["embeddings_matrix"]}')
    print(f'   Index  : {len(index_df)} entrées → {config["embeddings_index"]}')

    return embeddings_matrix, index_df


# Lancer l'encoding
embeddings_matrix, embeddings_index = encode_all_p1_audios(manifest_df, CONFIG)
print(f'\n📐 Shape finale : {embeddings_matrix.shape}  (N audios × 768 dims)')


🎙️  Encoding 4985 fichiers audio (batch=8)...


Batch encoding: 100%|██████████| 624/624 [2:19:54<00:00, 13.45s/it]  



✅ Encoding terminé !
   Matrix : (4985, 768)  → ../data/embeddings/audio_embeddings.npy
   Index  : 4985 entrées → ../data/embeddings/audio_embeddings_index.csv

📐 Shape finale : (4985, 768)  (N audios × 768 dims)


## ✅ CELLULE 7 — Enrichir pairs.csv avec les embeddings (pour P3)

In [7]:
# ════════════════════════════════════════════════════════════
# ENRICHISSEMENT pairs.csv
# Ajouter la colonne embedding_row dans pairs.csv pour P3
# ════════════════════════════════════════════════════════════

def enrich_pairs_with_embeddings(pairs_df: pd.DataFrame,
                                   embeddings_index: pd.DataFrame,
                                   output_path: str) -> pd.DataFrame:
    """
    Fusionne pairs.csv avec l'index des embeddings.
    Permet à P3 de retrouver directement le vecteur audio par son index.

    Returns:
        pairs enrichi avec colonne 'embedding_row'
    """
    if pairs_df.empty or embeddings_index.empty:
        print('⚠️  DataFrames vides — skip enrichissement')
        return pairs_df

    # Merge sur audio_id / filename
    emb_map = embeddings_index[['audio_id', 'embedding_row']].copy()

    # Extraire audio_id depuis audio_file (ex: cv_00001.wav → cv_00001)
    if 'audio_file' in pairs_df.columns:
        pairs_df['audio_id'] = pairs_df['audio_file'].apply(lambda x: Path(x).stem)

    enriched = pairs_df.merge(emb_map, on='audio_id', how='left')
    n_matched = enriched['embedding_row'].notna().sum()

    enriched.to_csv(output_path, index=False)
    print(f'✅ pairs enrichi : {n_matched}/{len(enriched)} paires avec embedding_row')
    print(f'   Sauvegardé → {output_path}')

    return enriched


# Enrichir tous les splits
for split_name, split_df in [('train', train_df), ('val', val_df), ('test', test_df)]:
    out = f'../data/output/pairs_{split_name}_with_embeddings.csv'
    enrich_pairs_with_embeddings(split_df, embeddings_index, out)

print('\n✅ P3 peut maintenant charger les embeddings avec :')
print('   emb = audio_embeddings[row_idx]  # vecteur (768,)')

✅ pairs enrichi : 3988/3988 paires avec embedding_row
   Sauvegardé → ../data/output/pairs_train_with_embeddings.csv
✅ pairs enrichi : 498/498 paires avec embedding_row
   Sauvegardé → ../data/output/pairs_val_with_embeddings.csv
✅ pairs enrichi : 499/499 paires avec embedding_row
   Sauvegardé → ../data/output/pairs_test_with_embeddings.csv

✅ P3 peut maintenant charger les embeddings avec :
   emb = audio_embeddings[row_idx]  # vecteur (768,)


## ✅ CELLULE 8 — SpeechEncoder class (pour P3 training)

In [8]:
class SpeechEncoder(nn.Module):
    """
    Module PyTorch à intégrer dans le Dual Encoder de P3.

    Usage P3:
        speech_enc = SpeechEncoder(frozen=True).to(device)
        emb = speech_enc(input_values)  # [B, 768]

    Note: passe le modèle wav2vec2 déjà chargé pour éviter
    de re-télécharger depuis HuggingFace.
    """
    def __init__(self,
                  pretrained_model=None,          # ← FIX : réutilise le modèle existant
                  model_name: str = 'facebook/wav2vec2-base-960h',
                  frozen: bool = True,
                  pooling: str = 'mean',
                  projection_dim: int = None):
        super().__init__()

        # Réutiliser le modèle déjà en mémoire si fourni
        if pretrained_model is not None:
            self.wav2vec2 = pretrained_model
            print('   ♻️  Modèle wav2vec2 réutilisé depuis mémoire (pas de re-download)')
        else:
            print(f'   ⏳ Chargement wav2vec2 depuis HF : {model_name}')
            self.wav2vec2 = Wav2Vec2Model.from_pretrained(model_name)

        self.pooling  = pooling

        if frozen:
            for p in self.wav2vec2.parameters():
                p.requires_grad = False
            print('   ❄️  Poids gelés (frozen=True)')

        self.projection = None
        self.out_dim = 768
        if projection_dim:
            self.projection = nn.Sequential(
                nn.Linear(768, projection_dim),
                nn.LayerNorm(projection_dim),
                nn.GELU()
            )
            self.out_dim = projection_dim
            print(f'   🔄 Projection : 768 → {projection_dim}')

    def forward(self, input_values: torch.Tensor,
                attention_mask: torch.Tensor = None) -> torch.Tensor:
        """ Input: [B, T] → Output: [B, out_dim] normalisé L2 """
        out = self.wav2vec2(input_values, attention_mask=attention_mask)
        h   = out.last_hidden_state  # [B, T, 768]

        if self.pooling == 'mean':
            if attention_mask is not None:
                m = attention_mask.unsqueeze(-1).float()
                emb = (h * m).sum(1) / m.sum(1).clamp(min=1e-8)
            else:
                emb = h.mean(1)
        else:
            emb = h.max(1).values

        if self.projection:
            emb = self.projection(emb)

        return nn.functional.normalize(emb, p=2, dim=-1)


# ── Test — réutilise le modèle déjà en mémoire ───────────────
print('🧪 Test SpeechEncoder...')
speech_encoder = SpeechEncoder(
    pretrained_model=model,    # ← modèle déjà chargé en cellule 4
    frozen=True
).to(DEVICE)

dummy = torch.randn(2, 16000 * 3).to(DEVICE)
with torch.no_grad():
    out = speech_encoder(dummy)

print(f'✅ SpeechEncoder : input {dummy.shape} → output {out.shape}')
print(f'   Norme L2 : {out.norm(dim=-1).mean().item():.4f} (≈1.0)')


🧪 Test SpeechEncoder...
   ♻️  Modèle wav2vec2 réutilisé depuis mémoire (pas de re-download)
   ❄️  Poids gelés (frozen=True)
✅ SpeechEncoder : input torch.Size([2, 48000]) → output torch.Size([2, 768])
   Norme L2 : 1.0000 (≈1.0)


## ✅ CELLULE 9 — Caching

In [9]:
class EmbeddingCache:
    def __init__(self, path='../data/embeddings/cache.pkl'):
        self.path  = path
        self._data = joblib.load(path) if os.path.exists(path) else {}
        print(f'💾 Cache : {len(self._data)} entrées')

    def _key(self, p):
        import hashlib
        s = os.stat(p)
        return hashlib.md5(f'{p}_{s.st_size}_{s.st_mtime}'.encode()).hexdigest()

    def get(self, p):  return self._data.get(self._key(p))
    def set(self, p, e): self._data[self._key(p)] = e
    def save(self):
        os.makedirs(os.path.dirname(self.path) or '.', exist_ok=True)
        joblib.dump(self._data, self.path)

cache = EmbeddingCache(CONFIG['cache_file'])
print('✅ Cache initialisé')

💾 Cache : 0 entrées
✅ Cache initialisé


## ✅ CELLULE 10 — Export ONNX

In [14]:
def export_to_onnx(output_path: str) -> bool:
    """Export le speech encoder en ONNX pour P4."""
    import torch.onnx
    os.makedirs(os.path.dirname(output_path) or '.', exist_ok=True)

    print(f'⏳ Export ONNX → {output_path}')

    dummy_wav = np.random.randn(16000 * 3).astype(np.float32)
    dummy_inp = processor(
        dummy_wav,
        sampling_rate=16000,
        return_tensors='pt'
    ).input_values.to('cpu')   # ONNX export toujours sur CPU

    # Copier le modèle sur CPU pour l'export
    model_cpu = Wav2Vec2Model.from_pretrained(CONFIG['model_name']).eval()

    try:
        torch.onnx.export(
            model_cpu,
            dummy_inp,
            output_path,
            input_names=['input_values'],
            output_names=['last_hidden_state'],
            dynamic_axes={
                'input_values'      : {0: 'batch', 1: 'seq'},
                'last_hidden_state' : {0: 'batch', 1: 'seq'}
            },
            opset_version=18,
            do_constant_folding=True,
        )
        del model_cpu  # libérer la RAM
        size_mb = os.path.getsize(output_path) / 1e6
        print(f'✅ ONNX exporté : {output_path} ({size_mb:.1f} MB)')
        return True
    except Exception as e:
        print(f'❌ Erreur ONNX : {e}')
        return False

# ⚠️  Décommenter uniquement en fin de projet (export lourd ~360MB)
export_to_onnx(CONFIG['onnx_path'])
print('✅ export_to_onnx() défini')
print('   → Décommenter la ligne ci-dessus pour lancer (étape finale)')


⏳ Export ONNX → ./models/speech_encoder.onnx


Loading weights: 100%|██████████| 210/210 [00:00<00:00, 2323.46it/s]
Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base-960h
Key               | Status     | 
------------------+------------+-
lm_head.weight    | UNEXPECTED | 
lm_head.bias      | UNEXPECTED | 
masked_spec_embed | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
W0321 15:47:40.208000 20376 Lib\site-packages\torch\onnx\_internal\exporter\_registration.py:110] torchvision is not installed. Skipping torchvision::nms
W0321 15:47:40.211000 20376 Lib\site-packages\torch\onnx\_internal\exporter\_registration.py:110] torchvision is not installed. Skipping torchvision::roi_align
W0321 15:47:40.224000 20376 Lib\site-packages\torch\onnx\_internal\exporter\_registration.py:110] torchvision is not installed. Skipping to

[torch.onnx] Obtain model graph for `Wav2Vec2Model([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `Wav2Vec2Model([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...
[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
Applied 15 of general pattern rewrite rules.
✅ ONNX exporté : ./models/speech_encoder.onnx (1.5 MB)
✅ export_to_onnx() défini
   → Décommenter la ligne ci-dessus pour lancer (étape finale)


## Vérifier que le fichier ONNX est utilisable

In [16]:
import onnxruntime as ort
import numpy as np

session = ort.InferenceSession('./models/speech_encoder.onnx')

# Test inférence
dummy = np.random.randn(1, 16000 * 3).astype(np.float32)
out = session.run(None, {'input_values': dummy})

print(f'✅ ONNX valide — output shape : {out[0].shape}')
# Attendu : (1, ~150, 768)

✅ ONNX valide — output shape : (1, 149, 768)


## ✅ CELLULE 11 — Tests système finaux

In [15]:
print('=' * 55)
print('🧪 TESTS SYSTÈME P2 (v2 — intégration P1)')
print('=' * 55)

passed = 0

# Audio synthétique pour tests — chemin cross-platform (Windows + Linux)
TEST_AUDIO_PATH = os.path.join(CONFIG['embeddings_dir'], 'test_audio.wav')
test_wav = np.random.randn(16000 * 3).astype(np.float32)
test_wav /= np.abs(test_wav).max()
sf.write(TEST_AUDIO_PATH, test_wav, 16000)
print(f'   Fichier test créé : {TEST_AUDIO_PATH}')

def check(name, cond, detail=''):
    global passed
    status = '✅' if cond else '❌'
    msg = f'{status}  {name}'
    if detail: msg += f'  [{detail}]'
    print(msg)
    if cond: passed += 1

# ── Test 1 : shape (768,) ────────────────────────────────────
e1 = speech_to_embedding(TEST_AUDIO_PATH)
check('Output shape = (768,)',
      e1 is not None and e1.shape == (768,),
      str(e1.shape) if e1 is not None else 'None')

# ── Test 2 : norme L2 ≈ 1.0 ─────────────────────────────────
check('Norme L2 ≈ 1.0',
      e1 is not None and abs(np.linalg.norm(e1) - 1.0) < 0.01,
      f'{np.linalg.norm(e1):.4f}' if e1 is not None else 'N/A')

# ── Test 3 : déterminisme ────────────────────────────────────
e2 = speech_to_embedding(TEST_AUDIO_PATH)
check('Déterminisme (même input → même output)',
      e1 is not None and e2 is not None and np.allclose(e1, e2, atol=1e-5))

# ── Test 4 : numpy array en entrée ───────────────────────────
e3 = speech_to_embedding(test_wav)
check('Audio numpy array → embedding OK',
      e3 is not None and e3.shape == (768,))

# ── Test 5 : SpeechEncoder batch ─────────────────────────────
dummy_batch = torch.randn(2, 16000 * 3).to(DEVICE)
with torch.no_grad():
    batch_out = speech_encoder(dummy_batch)
check('SpeechEncoder batch shape [2, 768]',
      batch_out.shape == (2, 768),
      str(batch_out.shape))

# ── Test 6 : matrice embeddings ──────────────────────────────
check('Embeddings matrix shape [N, 768]',
      embeddings_matrix.ndim == 2 and embeddings_matrix.shape[1] == 768,
      str(embeddings_matrix.shape))

TOTAL = 6
print('\n' + '=' * 55)
print(f'   {passed}/{TOTAL} tests passés')
if passed == TOTAL:
    print('   🎉 P2 v2 prête — intégration P1 validée !')
else:
    print(f'   ⚠️  {TOTAL - passed} test(s) échoué(s) — vérifier ci-dessus')
print('=' * 55)


🧪 TESTS SYSTÈME P2 (v2 — intégration P1)
   Fichier test créé : ../data/embeddings/test_audio.wav
✅  Output shape = (768,)  [(768,)]
✅  Norme L2 ≈ 1.0  [1.0000]
✅  Déterminisme (même input → même output)
✅  Audio numpy array → embedding OK
✅  SpeechEncoder batch shape [2, 768]  [torch.Size([2, 768])]
✅  Embeddings matrix shape [N, 768]  [(4985, 768)]

   6/6 tests passés
   🎉 P2 v2 prête — intégration P1 validée !


## 📋 Récap — Outputs P2 v2

| Fichier | Description | Consommé par |
|---|---|---|
| `audio_embeddings.npy` | Matrice [N, 768] de tous les audios | P3, P4 |
| `audio_embeddings_index.csv` | Mapping audio_id → row index | P3, P4 |
| `pairs_*_with_embeddings.csv` | Paires enrichies avec embedding_row | P3 |
| `speech_encoder.py` | Module importable | P3, P4 |
| `speech_encoder.onnx` | Export ONNX | P4 |

### 🔗 Utilisation par P3 (training)
```python
# Charger les embeddings pré-calculés (rapide)
embeddings = np.load('data/embeddings/audio_embeddings.npy')  # [N, 768]
index      = pd.read_csv('data/embeddings/audio_embeddings_index.csv')
pairs      = pd.read_csv('../data/output/pairs_train_with_embeddings.csv')

# Récupérer un embedding par row
row = pairs.iloc[0]['embedding_row']
emb = embeddings[int(row)]  # (768,)
```

### 🗓️ Planning
| Jour | Action |
|------|--------|
| **Jour 4** | Validation prétraitement — sync P1/P2/P3 |
| **Jour 6** | Lancer `encode_all_p1_audios()` sur dataset final |
| **Jour 7** | Livrer `audio_embeddings.npy` à P3 |
| **Jour 9** | Export ONNX → P4 |
